# Few-Shot (No CoT) Experiments

This notebook runs few-shot evaluation WITHOUT chain-of-thought on the same test sets used in the CoT experiments.
We use the same in-context examples as CoT but remove the reasoning steps, providing direct answers instead.

## 1. Setup & Imports

In [ ]:
import os
import sys
import pickle
import json
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
import time
from typing import Dict, List, Any, Tuple
from sklearn.model_selection import train_test_split
from IPython.display import display

# Add src to path
sys.path.append('src')

from models import ChatModel
from data_loading import create_dataset, create_cot_dataset, ensure_role_alternation
from parsing_utils import parse_response

# Set up results directory
RESULTS_DIR = Path("cache/few_shot_no_cot_experiments")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 2. Extract Test Data from Cache

First, run the extraction script to get test data from cached experiments:

In [12]:
# Run extraction script
!python scripts/extract_test_data.py

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Extracting test data from cached experiments...
Extracted 500 test samples from Qwen_Qwen2.5-1.5B-Instruct/sports_understanding
Extracted 500 test samples from Qwen_Qwen2.5-1.5B-Instruct/social_chemistry
Extracted 500 test samples from Qwen_Qwen2.5-1.5B-Instruct/logical_deduction
Extracted 500 test samples from Qwen_Qwen2.5-1.5B-Instruct/anachronisms
Extracted 500 test samples from google_gemma-2-2b-it/sports_understanding
Extracted 500 test samples from google_gemma-2-2b-it/social_chemistry
Extracted 500 test samples from google_gemma-2-2b-it/logical_deduction
Extracted 500 test samples from google_gemma-2-2b-it/anachronisms
Extracted 500 test samples from Qwen_Qwen2.5-3B-Instruct/sports_understanding
Extracted 500 test samples from Qwen_Qwen2.5-3B-Instruct/social_chemistry
Extracted 500 test samples from Qwen_Qwen2.5-3B-Instruct/logical_deduction
Extracted 500 test samples from Qwen_Qwen2.5-3B-Instruct/anachronisms
Extracted 500 test samples from deepseek-ai_DeepSeek-R1-Distill-Qwen-

In [13]:
# Load extracted test data
with open('cache/zero_shot_test_data.pkl', 'rb') as f:
    extracted_test_data = pickle.load(f)

print("Available models and datasets:")
for model_name, datasets in extracted_test_data.items():
    print(f"\n{model_name}:")
    for dataset_name, samples in datasets.items():
        print(f"  {dataset_name}: {len(samples)} samples")

Available models and datasets:

Qwen_Qwen2.5-1.5B-Instruct:
  sports_understanding: 500 samples
  social_chemistry: 500 samples
  logical_deduction: 500 samples
  anachronisms: 500 samples

google_gemma-2-2b-it:
  sports_understanding: 500 samples
  social_chemistry: 500 samples
  logical_deduction: 500 samples
  anachronisms: 500 samples

Qwen_Qwen2.5-3B-Instruct:
  sports_understanding: 500 samples
  social_chemistry: 500 samples
  logical_deduction: 500 samples
  anachronisms: 500 samples

deepseek-ai_DeepSeek-R1-Distill-Qwen-1.5B:
  sports_understanding: 500 samples
  social_chemistry: 500 samples
  logical_deduction: 500 samples
  anachronisms: 500 samples

google_gemma-2-9b-it:
  sports_understanding: 500 samples
  social_chemistry: 500 samples
  logical_deduction: 500 samples
  anachronisms: 500 samples

deepseek-ai_DeepSeek-R1-Distill-Qwen-7B:
  sports_understanding: 500 samples
  social_chemistry: 500 samples
  logical_deduction: 500 samples
  anachronisms: 500 samples

Qwen_Q

## 3. Few-Shot (No CoT) Evaluation Functions

In [ ]:
def create_strict_few_shot_prompt(sample: Dict[str, Any], dataset_name: str, model_name: str) -> List[Dict[str, str]]:
    """
    Create a few-shot prompt with strict formatting: responses must be exactly "A: (A)" or "A: (B)".
    """
    question = sample['question']
    choice_a = sample['choice_a']
    choice_b = sample['choice_b']
    
    # Create hardcoded few-shot examples for sports_understanding with strict formatting
    # These are manually crafted to ensure perfect format
    examples = [
        {
            'question': 'Is the following sentence plausible? "Trevor Bauer swung to protect the runner in the World Series"',
            'choice_a': 'No, the sentence is implausible',
            'choice_b': 'Yes, the sentence is plausible',
            'correct_letter': 'B'
        },
        {
            'question': 'Is the following sentence plausible? "Marouane Fellaini scored a freekick"',
            'choice_a': 'Yes, the sentence is plausible',
            'choice_b': 'No, the sentence is implausible',
            'correct_letter': 'A'
        },
        {
            'question': 'Is the following sentence plausible? "Javi Martinez committed a three second violation"',
            'choice_a': 'No, the sentence is implausible',
            'choice_b': 'Yes, the sentence is plausible',
            'correct_letter': 'A'
        },
        {
            'question': 'Is the following sentence plausible? "Juan Soto hit the slant pass in the NFC divisional round"',
            'choice_a': 'Yes, the sentence is plausible',
            'choice_b': 'No, the sentence is implausible',
            'correct_letter': 'B'
        }
    ]
    
    # Start with system instruction in first user message
    messages = []
    
    # First user message combines system instruction and first example
    first_example = examples[0]
    first_content = (
        f"Determine whether an artificially constructed sentence relating to sports is plausible or not.\n\n"
        f"Respond with only (A) or (B), nothing else.\n\n"
        f"Q: {first_example['question']}\n\n"
        f"Answer choices:\n"
        f"(A) {first_example['choice_a']}\n"
        f"(B) {first_example['choice_b']}"
    )
    messages.append({"role": "user", "content": first_content})
    messages.append({"role": "assistant", "content": f"A: ({first_example['correct_letter']})"})
    
    # Add remaining few-shot examples
    for example in examples[1:]:
        user_content = f"Q: {example['question']}\n\nAnswer choices:\n(A) {example['choice_a']}\n(B) {example['choice_b']}"
        messages.append({"role": "user", "content": user_content})
        messages.append({"role": "assistant", "content": f"A: ({example['correct_letter']})"})
    
    # Add the test question
    test_content = f"Q: {question}\n\nAnswer choices:\n(A) {choice_a}\n(B) {choice_b}"
    messages.append({"role": "user", "content": test_content})
    
    # Add assistant prefix to guide the model's response format
    is_deepseek = model_name and model_name.lower().startswith('deepseek')
    if not is_deepseek:
        messages.append({"role": "assistant", "content": "A: "})
    
    return messages


def evaluate_single_example(model: ChatModel, sample: Dict[str, Any], dataset_name: str,
                           temperature: float = 0.7, max_new_tokens: int = 10) -> Dict[str, Any]:
    """
    Evaluate a single example with strict few-shot (no CoT) prompting.
    """
    try:
        prompt = create_strict_few_shot_prompt(sample, dataset_name, model.model_name)
        prompt_str = model.apply_chat_template(prompt)
        
        # Generate response with minimal tokens since we expect only "A: (A)" or "A: (B)"
        generation = model.generate(prompt_str, temperature=temperature, max_new_tokens=max_new_tokens, do_sample=True)
        
        # Extract response - look for "A: (X)" pattern
        response = ""
        if "A:" in generation:
            # Find the last occurrence of "A:" which should be our response
            last_a_pos = generation.rfind("A:")
            after_a = generation[last_a_pos + 2:].strip()  # Skip "A:"
            
            # Look for (A) or (B) pattern after "A:"
            for pattern in ["(A)", "(B)"]:
                if pattern in after_a:
                    response = f"A: {pattern}"
                    break
        else:
            # Fallback: look for just (A) or (B) patterns and add "A:" prefix
            for pattern in ["(A)", "(B)"]:
                if pattern in generation:
                    response = f"A: {pattern}"
                    break
        
        # Parse response
        pred_letter, pred_answer = parse_response(response, thinking=False)
        
        # Check correctness
        is_correct = pred_letter == sample['correct_letter']
        
        return {
            'prompt': prompt_str,
            'response': response,
            'pred_letter': pred_letter,
            'pred_answer': pred_answer,
            'correct_letter': sample['correct_letter'],
            'correct_answer': sample['correct_answer'],
            'is_correct': is_correct
        }
    except Exception as e:
        return {
            'error': str(e),
            'response': '',
            'pred_letter': 'FAILED',
            'pred_answer': '',
            'correct_letter': sample['correct_letter'],
            'correct_answer': sample['correct_answer'],
            'is_correct': False
        }

## 4. Parallel Evaluation with Progress Tracking

In [ ]:
def evaluate_model_few_shot_no_cot(model_name: str, dataset_name: str, test_samples: List[Dict],
                                  max_workers: int = 8, max_samples: int = None) -> Dict[str, Any]:
    """
    Evaluate a model on few-shot (no CoT) test data with parallelization.
    """
    print(f"\nEvaluating {model_name} on {dataset_name}...")
    
    # Initialize model
    model = ChatModel(model_name)
    
    # Limit samples if requested
    if max_samples:
        test_samples = test_samples[:max_samples]
    
    results = []
    
    # Process in batches for better progress tracking
    batch_size = 10
    
    with tqdm(total=len(test_samples), desc=f"{dataset_name}") as pbar:
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            # Submit tasks in batches
            for i in range(0, len(test_samples), batch_size):
                batch = test_samples[i:i+batch_size]
                futures = []
                
                for sample in batch:
                    future = executor.submit(evaluate_single_example, model, sample, dataset_name)
                    futures.append(future)
                
                # Collect results from batch
                for future in as_completed(futures):
                    try:
                        result = future.result(timeout=30)
                        results.append(result)
                    except Exception as e:
                        print(f"Error processing example: {e}")
                        results.append({
                            'error': str(e),
                            'is_correct': False
                        })
                    pbar.update(1)
    
    # Calculate metrics
    correct_count = sum(1 for r in results if r.get('is_correct', False))
    total_count = len(results)
    accuracy = correct_count / total_count if total_count > 0 else 0
    
    # Count parse failures
    parse_failures = sum(1 for r in results if r.get('pred_letter') == 'FAILED')
    
    summary = {
        'model': model_name,
        'dataset': dataset_name,
        'total_samples': total_count,
        'correct': correct_count,
        'accuracy': accuracy,
        'parse_failures': parse_failures,
        'results': results
    }
    
    print(f"  Accuracy: {accuracy:.3f} ({correct_count}/{total_count})")
    print(f"  Parse failures: {parse_failures}")
    
    # Clean up model
    del model
    
    return summary

## 5. Run Experiments

### 5.1 Test with One Model/Dataset First

In [ ]:
# Test with one model and dataset first
test_cache_model = "google_gemma-2-2b-it"
test_actual_model = "google/gemma-2-2b-it"
test_dataset = "sports_understanding"

if test_cache_model in extracted_test_data and test_dataset in extracted_test_data[test_cache_model]:
    test_samples = extracted_test_data[test_cache_model][test_dataset]
    print(f"Testing {test_actual_model} on {test_dataset} with {len(test_samples[:10])} samples...")
    
    test_result = evaluate_model_few_shot_no_cot(
        test_actual_model, 
        test_dataset, 
        test_samples[:10],  # Just 10 samples for testing
        max_workers=2  # Reduced for testing
    )
    print(f"\nTest complete! Accuracy: {test_result['accuracy']:.3f}")
    
    # Show some examples
    print("\nExample responses:")
    for i, result in enumerate(test_result['results'][:3]):
        print(f"  Example {i+1}: {result['correct_letter']} -> {result['pred_letter']} ({'✓' if result['is_correct'] else '✗'})")
        print(f"    Response: '{result['response'][:100]}...'")
    
else:
    print(f"Test data not found for {test_cache_model}/{test_dataset}")

### 5.2 Run Full Experiments

In [ ]:
# Configure which models and datasets to evaluate
# Using cache-style model names (with underscores)
MODELS_TO_EVALUATE = [
    "Qwen_Qwen2.5-1.5B-Instruct",
    "Qwen_Qwen2.5-3B-Instruct", 
    "google_gemma-2-2b-it",
    "google_gemma-2-9b-it",
    "Qwen_Qwen2.5-7B-Instruct"
]

# Map cache names to actual model names for loading
MODEL_NAME_MAPPING = {
    "Qwen_Qwen2.5-1.5B-Instruct": "Qwen/Qwen2.5-1.5B-Instruct",
    "Qwen_Qwen2.5-3B-Instruct": "Qwen/Qwen2.5-3B-Instruct",
    "google_gemma-2-2b-it": "google/gemma-2-2b-it", 
    "google_gemma-2-9b-it": "google/gemma-2-9b-it",
    "Qwen_Qwen2.5-7B-Instruct": "Qwen/Qwen2.5-7B-Instruct"
}

DATASETS_TO_EVALUATE = [
    "sports_understanding",
    "anachronisms",
    "social_chemistry",
    "logical_deduction"
]

# Number of parallel workers (adjust based on available RAM)
MAX_WORKERS = 8

In [ ]:
# Run all experiments
all_results = {}

for cache_model_name in MODELS_TO_EVALUATE:
    model_results = {}
    
    # Skip if model not in extracted data
    if cache_model_name not in extracted_test_data:
        print(f"\nSkipping {cache_model_name} - no cached data found")
        continue
    
    # Get actual model name for loading
    actual_model_name = MODEL_NAME_MAPPING[cache_model_name]
    
    for dataset_name in DATASETS_TO_EVALUATE:
        # Skip if dataset not available for this model
        if dataset_name not in extracted_test_data[cache_model_name]:
            print(f"\nSkipping {cache_model_name}/{dataset_name} - no cached data found")
            continue
        
        test_samples = extracted_test_data[cache_model_name][dataset_name]
        
        # Run evaluation using actual model name
        start_time = time.time()
        result = evaluate_model_few_shot_no_cot(
            actual_model_name,
            dataset_name,
            test_samples,
            max_workers=MAX_WORKERS
        )
        elapsed_time = time.time() - start_time
        result['elapsed_time'] = elapsed_time
        result['cache_model_name'] = cache_model_name  # Track cache name
        
        model_results[dataset_name] = result
        
        # Save intermediate results using cache name
        save_path = RESULTS_DIR / f"{cache_model_name}_{dataset_name}_results.pkl"
        with open(save_path, 'wb') as f:
            pickle.dump(result, f)
        
        print(f"  Time: {elapsed_time:.1f}s")
        print(f"  Saved to: {save_path}")
    
    all_results[cache_model_name] = model_results

print("\n" + "="*50)
print("All experiments complete!")

## 6. Analysis & Comparison with CoT Results

In [ ]:
# Load CoT results for comparison
from src.results import Results

cot_results = Results("cache/experiments")
cot_summary = cot_results.get_summary()

In [ ]:
# Create comparison table
comparison_data = []

for cache_model_name, model_results in all_results.items():
    actual_model_name = MODEL_NAME_MAPPING[cache_model_name]
    
    for dataset_name, result in model_results.items():
        # Get CoT accuracy using actual model name
        try:
            cot_acc = cot_results.generation.get_accuracy(
                model=actual_model_name,
                dataset=dataset_name,
                split='test'
            ) / 100  # Convert from percentage
        except:
            cot_acc = None
        
        comparison_data.append({
            'Model': cache_model_name.replace('_', '/').replace('Qwen/Qwen2.5', 'Qwen2.5').replace('google/', ''),  # Prettier name
            'Dataset': dataset_name,
            'Few-Shot (No CoT) Acc': result['accuracy'],
            'CoT Acc': cot_acc,
            'Difference': result['accuracy'] - cot_acc if cot_acc else None,
            'Samples': result['total_samples']
        })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.round(3)

In [ ]:
# Display comparison table
print("\nFew-Shot (No CoT) vs CoT Accuracy Comparison:")
print("="*70)
display(comparison_df)

In [ ]:
# Create summary statistics
print("\nSummary Statistics:")
print("="*50)

# Average by model
model_avg = comparison_df.groupby('Model')[['Few-Shot (No CoT) Acc', 'CoT Acc', 'Difference']].mean()
print("\nAverage Accuracy by Model:")
display(model_avg.round(3))

# Average by dataset
dataset_avg = comparison_df.groupby('Dataset')[['Few-Shot (No CoT) Acc', 'CoT Acc', 'Difference']].mean()
print("\nAverage Accuracy by Dataset:")
display(dataset_avg.round(3))

# Overall statistics
overall_few_shot_no_cot = comparison_df['Few-Shot (No CoT) Acc'].mean()
overall_cot = comparison_df['CoT Acc'].mean()
overall_diff = comparison_df['Difference'].mean()

print(f"\nOverall Average:")
print(f"  Few-Shot (No CoT): {overall_few_shot_no_cot:.3f}")
print(f"  CoT: {overall_cot:.3f}")
print(f"  Difference: {overall_diff:.3f}")

In [ ]:
# Save comparison to CSV
comparison_df.to_csv(RESULTS_DIR / "few_shot_no_cot_vs_cot_comparison.csv", index=False)
print(f"\nComparison saved to: {RESULTS_DIR / 'few_shot_no_cot_vs_cot_comparison.csv'}")

## 7. Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Create comparison bar plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Accuracy comparison by model
model_comparison = comparison_df.groupby('Model')[['Few-Shot (No CoT) Acc', 'CoT Acc']].mean()
model_comparison.plot(kind='bar', ax=axes[0], width=0.8)
axes[0].set_title('Average Accuracy by Model', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Model', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_ylim([0, 1])
axes[0].legend(['Few-Shot (No CoT)', 'CoT'], loc='lower right')
axes[0].grid(axis='y', alpha=0.3)

# Rotate x labels
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')

# Plot 2: Accuracy comparison by dataset
dataset_comparison = comparison_df.groupby('Dataset')[['Few-Shot (No CoT) Acc', 'CoT Acc']].mean()
dataset_comparison.plot(kind='bar', ax=axes[1], width=0.8)
axes[1].set_title('Average Accuracy by Dataset', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Dataset', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_ylim([0, 1])
axes[1].legend(['Few-Shot (No CoT)', 'CoT'], loc='lower right')
axes[1].grid(axis='y', alpha=0.3)

# Rotate x labels
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'few_shot_no_cot_vs_cot_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Plot saved to: {RESULTS_DIR / 'few_shot_no_cot_vs_cot_comparison.png'}")

In [ ]:
# Create heatmap of differences
pivot_diff = comparison_df.pivot(index='Model', columns='Dataset', values='Difference')

plt.figure(figsize=(10, 6))
sns.heatmap(pivot_diff, annot=True, fmt='.3f', cmap='RdBu_r', center=0, 
            cbar_kws={'label': 'Difference (Few-Shot No CoT - CoT)'})
plt.title('Accuracy Difference: Few-Shot (No CoT) - CoT', fontsize=14, fontweight='bold')
plt.xlabel('Dataset', fontsize=12)
plt.ylabel('Model', fontsize=12)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'accuracy_difference_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Heatmap saved to: {RESULTS_DIR / 'accuracy_difference_heatmap.png'}")

## 8. Save Final Results

In [ ]:
# Save all results
final_results = {
    'experiments': all_results,
    'comparison': comparison_df.to_dict('records'),
    'summary': {
        'overall_few_shot_no_cot_accuracy': overall_few_shot_no_cot,
        'overall_cot_accuracy': overall_cot,
        'overall_difference': overall_diff,
        'by_model': model_avg.to_dict(),
        'by_dataset': dataset_avg.to_dict()
    }
}

with open(RESULTS_DIR / 'few_shot_no_cot_final_results.pkl', 'wb') as f:
    pickle.dump(final_results, f)

with open(RESULTS_DIR / 'few_shot_no_cot_summary.json', 'w') as f:
    json.dump(final_results['summary'], f, indent=2)

print(f"Results saved to:")
print(f"  - {RESULTS_DIR / 'few_shot_no_cot_final_results.pkl'}")
print(f"  - {RESULTS_DIR / 'few_shot_no_cot_summary.json'}")